# YOLO Inference — Raw vs Preprocessed Comparison
Runs inference on every image in a folder and shows side-by-side detections on the raw and preprocessed versions.

In [ ]:
%pip install -q ultralytics matplotlib pillow opencv-python

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from preprocessing import DocumentPreprocessor

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH = Path("best_iou_median.pt")
FOLDER = Path("yolo_dataset/images")  # ← change to your image folder
CONF = 0.25  # detection confidence threshold
IOU = 0.45  # NMS IoU threshold
DISPLAY_H = 700  # display height per panel (px)
SAVE_DIR = Path("inference_compare")  # set to None to skip saving to disk
# ─────────────────────────────────────────────────────────────────────────────

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

In [ ]:
model = YOLO(str(MODEL_PATH))
preprocessor = DocumentPreprocessor()
print(f"Model loaded. Folder: {FOLDER.resolve()}")

In [ ]:
BOX_COLOR_RAW = (50, 50, 220)  # BGR  →  red-ish in display
BOX_COLOR_PREP = (30, 160, 30)  # BGR  →  green in display


def draw_boxes(img_bgr, boxes, names, color_bgr):
    """Draw YOLO boxes on a BGR image, return BGR."""
    out = img_bgr.copy()
    for b in boxes:
        x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
        conf = float(b.conf.item())
        cls_id = int(b.cls.item())
        cv2.rectangle(out, (x1, y1), (x2, y2), color_bgr, 2)
        label = f"{names[cls_id]} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        ty = max(y1 - 4, th + 4)
        cv2.rectangle(out, (x1, ty - th - 4), (x1 + tw + 4, ty), color_bgr, -1)
        cv2.putText(
            out,
            label,
            (x1 + 2, ty - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )
    return out


def resize_h(img, h):
    scale = h / img.shape[0]
    return cv2.resize(
        img, (max(1, int(img.shape[1] * scale)), h), interpolation=cv2.INTER_AREA
    )


def pad_to_h(img, h):
    if img.shape[0] >= h:
        return img[:h]
    pad = np.full((h - img.shape[0], img.shape[1], 3), 255, dtype=np.uint8)
    return np.vstack([img, pad])

In [ ]:
# Resolve preprocessed directory (created by: python preprocessing.py <FOLDER>)
preprocessed_dir = FOLDER.parent / f"{FOLDER.name}_preprocessed"
if not preprocessed_dir.exists():
    preprocessed_dir = FOLDER / "preprocessed"  # fallback: subfolder

images = sorted(p for p in FOLDER.iterdir() if p.suffix.lower() in IMAGE_EXTS)
print(f"Found {len(images)} images in {FOLDER}")
print(f"Preprocessed dir : {preprocessed_dir}  (exists={preprocessed_dir.exists()})")

if SAVE_DIR:
    SAVE_DIR.mkdir(exist_ok=True)

for img_path in images:
    # ── raw image ──
    raw_bgr = cv2.imread(str(img_path))
    if raw_bgr is None:
        print(f"  [skip] {img_path.name} — unreadable")
        continue

    # ── preprocessed image ──
    prep_path = preprocessed_dir / f"{img_path.stem}.jpg"
    if not prep_path.exists():
        print(f"  [skip] {img_path.name} — preprocessed not found ({prep_path})")
        print(f"         Run first:  python preprocessing.py {FOLDER}")
        continue
    prep_bgr = cv2.imread(str(prep_path))

    # ── run inference on both ──
    raw_res = model.predict(str(img_path), conf=CONF, iou=IOU, verbose=False)[0]
    prep_res = model.predict(str(prep_path), conf=CONF, iou=IOU, verbose=False)[0]
    names = raw_res.names

    n_raw = len(raw_res.boxes)
    n_prep = len(prep_res.boxes)
    delta = n_prep - n_raw
    sign = "+" if delta >= 0 else ""
    print(
        f"  {img_path.name:35s}  raw={n_raw:3d}  preprocessed={n_prep:3d}  ({sign}{delta})"
    )

    # ── annotate ──
    raw_ann = draw_boxes(raw_bgr, raw_res.boxes, names, BOX_COLOR_RAW)
    prep_ann = draw_boxes(prep_bgr, prep_res.boxes, names, BOX_COLOR_PREP)

    # ── side-by-side display ──
    raw_disp = resize_h(raw_ann, DISPLAY_H)
    prep_disp = resize_h(prep_ann, DISPLAY_H)
    h = max(raw_disp.shape[0], prep_disp.shape[0])
    combined_bgr = np.hstack([pad_to_h(raw_disp, h), pad_to_h(prep_disp, h)])

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    axes[0].imshow(raw_ann[:, :, ::-1])
    axes[0].set_title(f"Raw  —  {n_raw} detections", fontsize=13, color="#cc3333")
    axes[0].axis("off")
    axes[1].imshow(prep_ann[:, :, ::-1])
    axes[1].set_title(
        f"Preprocessed  —  {n_prep} detections", fontsize=13, color="#1e901e"
    )
    axes[1].axis("off")
    fig.suptitle(img_path.name, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

    if SAVE_DIR:
        save_path = SAVE_DIR / f"{img_path.stem}_compare.jpg"
        Image.fromarray(combined_bgr[:, :, ::-1]).save(str(save_path))